# PDF Keyword + Appeal-Scope Harvester (ET Corpus → _Matches)

This script scans a large folder of ET-case PDFs and builds a **targeted "matches" library** for manual review and downstream analysis.

---

## Purpose

Turn a large, unstructured PDF corpus into a structured shortlist of potentially relevant cases using deterministic keyword and regex filtering.

This is a **high-speed recall harvester**, not a semantic or LLM-based precision layer.

---

## Step 1 — Crawl and Pre-Filter PDFs

* Recursively finds all `*.pdf` files under `INPUT_ROOT`.
* Rejects small documents (`pages < MIN_PAGES`).
* Extracts text from the first `TEXT_PAGES_TO_SCAN` pages (cheap front-scan).

This keeps processing fast while avoiding irrelevant small files.

---

## Step 2 — Two-Tier Matching Logic

### Gate Condition (Mandatory)

Every PDF must contain **all phrases in `NEEDLES_ALL`**.

If this fails → the document is ignored.

### Keep Condition (At Least One Required)

After passing the gate, the PDF must satisfy **at least one** of the following:

* Contain any phrase from `NEEDLES_ANY` (simple substring match), OR
* Match `APPEAL_SCOPE_REGEX` (detects appeal-scope limitation language such as:

  * "not raised in the appeal"
  * "outside the scope of the appeal"
  * "declined to consider"
  * "failed to engage with"
  * etc.)

Only documents passing Gate + Keep are retained.

---

## Step 3 — Parallel Scanning

* Uses `ProcessPoolExecutor` with up to `MAX_WORKERS` processes.
* Each PDF is scanned independently.

For matched files, metadata collected includes:

* `path`
* `pages`
* `size_mb`
* `mtime`
* `hit_all`
* `hit_any`
* `appeal_scope_hit`
* `appeal_scope_match` (small snippet of matched phrase)

---

## Step 4 — Structured Copy to Matches Folder

All matched PDFs are copied into `MATCHES_ROOT`.

### Folder Structure

* One folder per `NEEDLES_ANY` term (slugified)
* One dedicated folder `_APPEAL_SCOPE_REGEX` for all regex hits

If `PRESERVE_STRUCTURE = True`, original directory hierarchy is preserved under each subfolder.

This prevents filename collisions and preserves provenance.

---

## Step 5 — Master Index CSV

Exactly one CSV is written to:

`MATCHES_ROOT/_matches_index.csv`

The CSV contains:

* File metadata
* Raw hit indicators
* Boolean columns per NEEDLES_ANY (e.g., `has__predetermination`)
* Boolean column for regex hits (`has__appeal_scope_regex`)
* Root path reference

---

## Output Artifacts

1. Curated PDF library under `MATCHES_ROOT/`
2. Single master index CSV for analysis

---

## Architectural Position

This module is a **deterministic recall harvester**.

It does not perform:

* Semantic analysis
* LLM reasoning
* Paragraph-level anchoring

Its role is to reduce a massive PDF corpus into a manageable, topic-filtered working set for deeper analysis (e.g., Moltie or WBerious precision layers).

---

## Guiding Principle

Fast recall first. Precision and reasoning later.


In [1]:
import shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import fitz  # PyMuPDF
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

INPUT_ROOT = Path(r"/media/hello/Vault/Tribunals/ET_Cases/").resolve()

# Principal matches folder (many subfolders live under here)
MATCHES_ROOT = Path(r"/media/hello/Vault/Tribunals/_Matches").resolve()

NEEDLES_ALL = [
    "unfair dismissal",
]

NEEDLES_ANY = [
    "upheld",
    "verbal warning",
    "no contemporaneous evidence",
    "predetermination",
]
import re

APPEAL_SCOPE_REGEX = re.compile(
    r"""
    (
        (not\s+raised\s+(?:in|within)\s+the\s+(?:written\s+)?appeal) |
        (outside\s+the\s+scope\s+of\s+the\s+appeal) |
        (declined\s+to\s+consider) |
        (refused\s+to\s+consider) |
        (limited\s+to\s+the\s+grounds) |
        (confined\s+to\s+(?:the\s+)?grounds) |
        (new\s+grounds\s+(?:raised|introduced)\s+(?:at|during)\s+the\s+appeal) |
        (raised\s+at\s+the\s+appeal\s+hearing) |
        (fresh\s+consideration) |
        (rubber\s+stamp) |
        (failed\s+to\s+engage\s+with) |
        (did\s+not\s+address\s+the\s+appeal\s+ground)
    )
    """,
    re.IGNORECASE | re.VERBOSE
)
CASE_SENSITIVE = False

MIN_PAGES = 4
TEXT_PAGES_TO_SCAN = 12
MAX_WORKERS = 24
PRESERVE_STRUCTURE = True

# Single CSV written ONLY in MATCHES_ROOT
MASTER_CSV_NAME = "_matches_index.csv"


# =========================================================
# HELPERS
# =========================================================

def _norm(s: str) -> str:
    return s if CASE_SENSITIVE else s.lower()


def _slugify(s: str) -> str:
    s = s.strip().replace(" ", "_")
    s = "".join(ch for ch in s if ch.isalnum() or ch in ("_", "-", "."))
    return s[:120] if s else "EMPTY"


def iter_pdfs(root: Path):
    for p in root.rglob("*.pdf"):
        if p.is_file():
            yield p


def _safe_copy(src: Path, dst: Path) -> Path:
    """
    Copy src to dst. If dst exists, add suffix _1, _2, ...
    Returns final path.
    """
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
        return dst

    stem, suffix = dst.stem, dst.suffix
    i = 1
    while True:
        cand = dst.with_name(f"{stem}_{i}{suffix}")
        if not cand.exists():
            shutil.copy2(src, cand)
            return cand
        i += 1


def scan_one(pdf_path: str):
    """
    One-pass scan:
      - page count filter
      - extract first N pages text
      - require NEEDLES_ALL (all must be present)
      - compute which NEEDLES_ANY are present (simple substring)
      - ALSO detect appeal-scope limitation via APPEAL_SCOPE_REGEX
    Returns row dict (with hits) or None.
    """
    p = Path(pdf_path)
    try:
        stat = p.stat()
        size_mb = stat.st_size / (1024 * 1024)

        doc = fitz.open(p)
        pages = doc.page_count
        if pages < MIN_PAGES:
            doc.close()
            return None

        n = min(TEXT_PAGES_TO_SCAN, pages)
        hay = []
        for i in range(n):
            try:
                hay.append(doc.load_page(i).get_text("text"))
            except Exception:
                pass
        doc.close()

        text = "\n".join(hay)
        text_n = _norm(text)

        needles_all = [_norm(x) for x in NEEDLES_ALL if x and x.strip()]
        needles_any = [_norm(x) for x in NEEDLES_ANY if x and x.strip()]

        # 1) MUST satisfy NEEDLES_ALL (gate)
        ok_all = all(k in text_n for k in needles_all) if needles_all else True
        if not ok_all:
            return None

        # 2) substring hits
        hit_any = [k for k in needles_any if k in text_n]

        # 3) regex hit (appeal scope)
        # IMPORTANT: APPEAL_SCOPE_REGEX should be compiled globally (re.IGNORECASE)
        m = APPEAL_SCOPE_REGEX.search(text)  # use raw `text` since regex is IGNORECASE
        appeal_scope_hit = bool(m)
        appeal_scope_match = (m.group(0)[:250] if m else "")

        # keep doc if it hits at least one NEEDLES_ANY OR regex hit
        if (not hit_any) and (not appeal_scope_hit):
            return None

        return {
            "path": str(p),
            "pages": pages,
            "size_mb": round(size_mb, 3),
            "mtime": pd.to_datetime(stat.st_mtime, unit="s"),
            "hit_all": "; ".join([k for k in needles_all if k in text_n]),
            "hit_any": "; ".join(hit_any),
            "appeal_scope_hit": appeal_scope_hit,
            "appeal_scope_match": appeal_scope_match,  # small snippet of the matched phrase
        }

    except Exception:
        return None


def copy_to_subfolder(src: Path, in_root: Path, subfolder: Path, preserve_structure: bool = True) -> Path:
    """
    Copy one PDF into subfolder, optionally preserving structure.
    Returns final copied path.
    """
    if preserve_structure:
        try:
            rel = src.relative_to(in_root)
        except ValueError:
            rel = Path(src.name)
        dst = subfolder / rel
    else:
        dst = subfolder / src.name

    return _safe_copy(src, dst)


# =========================================================
# MAIN
# =========================================================

def main():
    MATCHES_ROOT.mkdir(parents=True, exist_ok=True)

    pdfs = list(iter_pdfs(INPUT_ROOT))
    rows = []

    print(f"[scan] Input root: {INPUT_ROOT}")
    print(f"[scan] PDFs found: {len(pdfs)}")
    print(f"[scan] NEEDLES_ALL (must match all): {NEEDLES_ALL}")
    print(f"[scan] NEEDLES_ANY (substring OR regex): {NEEDLES_ANY}")
    print(f"[scan] Pages >= {MIN_PAGES}, scanning first {TEXT_PAGES_TO_SCAN} pages")
    print(f"[out] Principal matches folder: {MATCHES_ROOT}")
    print(f"[out] Preserve structure: {PRESERVE_STRUCTURE}")

    # 1) Scan once in parallel
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(scan_one, str(p)) for p in pdfs]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="Scanning PDFs", unit="file"):
            r = fut.result()
            if r is not None:
                rows.append(r)

    df = pd.DataFrame(rows)
    if df.empty:
        print("[scan] Matches: 0")
        return df

    df = df.sort_values(["pages", "size_mb"], ascending=False).reset_index(drop=True)
    print(f"[scan] Matches (NEEDLES_ANY OR APPEAL_SCOPE_REGEX): {len(df)}")

    # 2) Create one subfolder per NEEDLES_ANY term and copy files into each
    needles_any_norm = [_norm(x) for x in NEEDLES_ANY if x and x.strip()]
    norm_to_original = {_norm(x): x for x in NEEDLES_ANY if x and x.strip()}

    # For routing, explode hit_any into a list (may be empty if only regex hit)
    df["hit_any_list"] = df.get("hit_any", "").fillna("").apply(
        lambda s: [x.strip() for x in s.split(";") if x.strip()]
    )

    for needle_norm in needles_any_norm:
        label = norm_to_original[needle_norm]
        folder_name = _slugify(label)
        out_dir = (MATCHES_ROOT / folder_name).resolve()
        out_dir.mkdir(parents=True, exist_ok=True)

        mask = df["hit_any_list"].apply(lambda xs: needle_norm in xs)
        df_group = df[mask]

        if df_group.empty:
            print(f"[group] '{label}' -> 0 matches (skip)")
            continue

        print(f"[group] '{label}' -> {len(df_group)} matches -> {out_dir}")

        for src_str in tqdm(df_group["path"].tolist(), desc=f"Copying -> {folder_name}", unit="file"):
            src = Path(src_str)
            copy_to_subfolder(
                src=src,
                in_root=INPUT_ROOT,
                subfolder=out_dir,
                preserve_structure=PRESERVE_STRUCTURE,
            )

    # 2b) Copy ALL regex hits into ONE dedicated folder
    REGEX_FOLDER_NAME = "_APPEAL_SCOPE_REGEX"
    if "appeal_scope_hit" in df.columns:
        df_regex = df[df["appeal_scope_hit"] == True].copy()
    else:
        df_regex = df.iloc[0:0].copy()

    print(f"[regex] APPEAL_SCOPE_REGEX hits: {len(df_regex)}")

    if not df_regex.empty:
        out_dir = (MATCHES_ROOT / REGEX_FOLDER_NAME).resolve()
        out_dir.mkdir(parents=True, exist_ok=True)

        for src_str in tqdm(df_regex["path"].tolist(), desc=f"Copying -> {REGEX_FOLDER_NAME}", unit="file"):
            src = Path(src_str)
            copy_to_subfolder(
                src=src,
                in_root=INPUT_ROOT,
                subfolder=out_dir,
                preserve_structure=PRESERVE_STRUCTURE,
            )

    # 3) Write ONE master CSV in MATCHES_ROOT (and nowhere else)
    df2 = df.drop(columns=["hit_any_list"], errors="ignore").copy()
    df2["matches_root"] = str(MATCHES_ROOT)

    # boolean columns per NEEDLES_ANY
    for needle_norm in needles_any_norm:
        label = norm_to_original[needle_norm]
        col = f"has__{_slugify(label)}"
        df2[col] = df.get("hit_any", "").fillna("").apply(
            lambda s: needle_norm in [x.strip() for x in s.split(";") if x.strip()]
        )

    # boolean column for regex
    if "appeal_scope_hit" in df2.columns:
        df2["has__appeal_scope_regex"] = df2["appeal_scope_hit"].fillna(False).astype(bool)
    else:
        df2["has__appeal_scope_regex"] = False

    master_path = MATCHES_ROOT / MASTER_CSV_NAME
    df2.to_csv(master_path, index=False, quoting=1)  # csv.QUOTE_ALL = 1
    print(f"[csv] Wrote single master CSV: {master_path}")

    return df2


if __name__ == "__main__":
    df = main()
    print(df.head(25))

[scan] Input root: /media/hello/Vault/Tribunals/ET_Cases
[scan] PDFs found: 127755
[scan] NEEDLES_ALL (must match all): ['unfair dismissal']
[scan] NEEDLES_ANY (substring OR regex): ['upheld', 'verbal warning', 'no contemporaneous evidence', 'predetermination']
[scan] Pages >= 4, scanning first 12 pages
[out] Principal matches folder: /media/hello/Vault/Tribunals/_Matches
[out] Preserve structure: True


Scanning PDFs: 100%|██████████| 127755/127755 [00:17<00:00, 7343.06file/s]


[scan] Matches (NEEDLES_ANY OR APPEAL_SCOPE_REGEX): 2859
[group] 'upheld' -> 2544 matches -> /media/hello/Vault/Tribunals/_Matches/upheld


Copying -> upheld: 100%|██████████| 2544/2544 [00:00<00:00, 5201.38file/s]


[group] 'verbal warning' -> 251 matches -> /media/hello/Vault/Tribunals/_Matches/verbal_warning


...

[group] 'predetermination' -> 36 matches -> /media/hello/Vault/Tribunals/_Matches/predetermination


Copying -> predetermination: 100%|██████████| 36/36 [00:00<00:00, 4544.21file/s]


[regex] APPEAL_SCOPE_REGEX hits: 150


Copying -> _APPEAL_SCOPE_REGEX: 100%|██████████| 150/150 [00:00<00:00, 5139.83file/s]


[csv] Wrote single master CSV: /media/hello/Vault/Tribunals/_Matches/_matches_index.csv
                                                 path  pages  size_mb  \
0   /media/hello/Vault/Tribunals/ET_Cases/Ms_B_v_M...    264    2.136   
1   /media/hello/Vault/Tribunals/ET_Cases/1._Mr_M_...    197    1.684   
2   /media/hello/Vault/Tribunals/ET_Cases/Mr_G_Gal...    185    1.137   
3   /media/hello/Vault/Tribunals/ET_Cases/Christop...    172    1.495   
4   /media/hello/Vault/Tribunals/ET_Cases/Mr_A_Ham...    171    2.749   
5   /media/hello/Vault/Tribunals/ET_Cases/Ms_T-J_H...    151    1.354   
6   /media/hello/Vault/Tribunals/ET_Cases/Mr_Z_Yan...    138    1.014   
7   /media/hello/Vault/Tribunals/ET_Cases/Mr_J_Whi...    137    1.566   
8   /media/hello/Vault/Tribunals/ET_Cases/Mrs_L_Pr...    137    0.920   
9   /media/hello/Vault/Tribunals/ET_Cases/Ms_L_Lew...    132    0.992   
10  /media/hello/Vault/Tribunals/ET_Cases/Ms_P_M_C...    130    4.918   
11  /media/hello/Vault/Tribunals/ET_